# Day 22 — CNN Fundamentals

## 1. Learning Objectives
- Understand the problem with Linear Layers for images.
- Understand Convolutions, Kernels (Filters), and Feature Maps.
- Learn how Stride and Padding affect spatial dimensions.
- Understand Pooling (Max Pooling).
- Build visual and mathematical intuition for CNNs.

## 2. Prerequisites
- Tensor Shapes (Day 4: `[Batch, Channels, Height, Width]`)
- Experience flattening images (Day 21).

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

## 3. Concept Explanation: The Problem with `nn.Linear`
If you have a 1000x1000 pixel image, it has 1,000,000 pixels. A single `nn.Linear(1000000, 1024)` layer requires **1 BILLION weights**. It will instantly run out of memory.
Worse, if you flatten a 2D image into a 1D vector, the network doesn't know that a pixel at index `0` is physically right next to the pixel at index `1000`. Spatial structure is destroyed.

**Convolutional Neural Networks (CNNs)** solve this by sliding a small magnifying glass (a **Kernel** or **Filter**) over the image to detect local patterns (edges, corners, textures).

## 5. Intuition & Visual Model
- **Kernel / Filter**: A small matrix (usually 3x3) of weights.
- **Convolution**: The process of sliding this 3x3 kernel across the image, doing element-wise multiplication, and summing the result into a single pixel on a new image.
- **Feature Map**: The new "image" created by the convolution. It acts as a map showing *where* the kernel found its specific pattern.

## 6. Mathematical Foundation: The Convolution Operation
Let's simulate a convolution operation manually without `nn.Conv2d`.

In [ ]:
# 8. Simple Example
# Let's create a 5x5 "Image" (e.g., a vertical line of 1s in the middle)
image = torch.tensor([
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0]
], dtype=torch.float32)

# Let's create a 3x3 "Vertical Edge Detector" Kernel
kernel = torch.tensor([
    [-1, 2, -1],
    [-1, 2, -1],
    [-1, 2, -1]
], dtype=torch.float32)

# PyTorch's functional conv2d requires shape [Batch, Channels, Height, Width]
img_tensor = image.view(1, 1, 5, 5)
kernel_tensor = kernel.view(1, 1, 3, 3)

# Perform the Convolution
feature_map = F.conv2d(img_tensor, kernel_tensor)

print("Original Image (5x5):\n", image)
print("\nKernel (3x3):\n", kernel)
print("\nFeature Map (3x3) - Notice how it highlights the middle!\n", feature_map.squeeze())

## 9. Code Walkthrough: Padding and Stride
Notice that our 5x5 image became a 3x3 feature map. Why? Because a 3x3 kernel can only fit inside a 5x5 grid 3 times horizontally and 3 times vertically. 

- **Padding**: Adding zeros around the border of the original image so the kernel can slide over the edges. This preserves the original spatial dimensions.
- **Stride**: How many pixels the kernel shifts at a time. Default is 1. A stride of 2 will skip every other pixel, effectively halving the image size.

In [ ]:
# Apply padding=1 (adds a 1-pixel border of zeros)
padded_feat_map = F.conv2d(img_tensor, kernel_tensor, padding=1)
print("Shape with Padding=1:", padded_feat_map.shape) # Stays 5x5!

# Apply stride=2 (jumps by 2 pixels)
strided_feat_map = F.conv2d(img_tensor, kernel_tensor, padding=1, stride=2)
print("Shape with Stride=2:", strided_feat_map.shape) # Shrinks to 3x3!

## 10. Pooling (Max Pooling)
Pooling is a way to aggressively downsample an image. It takes a window (e.g., 2x2) and keeps only the maximum value in that window. 
- It drastically reduces parameters.
- It provides **translation invariance** (if an eye is slightly to the left, max pooling still detects it).

In [ ]:
big_image = torch.randn(1, 1, 4, 4)
pooled = F.max_pool2d(big_image, kernel_size=2, stride=2)

print("Original Shape:", big_image.shape)
print("Pooled Shape:", pooled.shape) # Cut exactly in half!

## 11. Practice Exercise 1: Dimension Math
You have an input image of shape `[1, 3, 32, 32]`. 
You apply a `3x3` kernel, with `padding=0` and `stride=1`. 
What is the Height and Width of the resulting feature map?

**Solution:**
Formula: $Output = \lfloor \frac{Input - Kernel + 2 \times Padding}{Stride} \rfloor + 1$
Output = (32 - 3 + 0) / 1 + 1 = 29 + 1 = 30.
The result is `[1, C, 30, 30]`.

## 13. Debugging Challenge
Why does the following Max Pooling operation crash?

In [ ]:
my_image = torch.randn(28, 28) # A 28x28 grayscale image
# result = F.max_pool2d(my_image, kernel_size=2, stride=2) # Uncomment to see error

**Solution:** `F.max_pool2d` and `F.conv2d` strictly require 4-dimensional tensors: `[Batch, Channels, Height, Width]`. `my_image` is only 2D. 
Fix: `my_image = my_image.view(1, 1, 28, 28)`.

## 17. Interview Questions
1. **Why do we use Convolutional layers instead of Linear layers for images?**
   *Answer*: CNNs have two massive advantages: 
   1) **Parameter Sharing**: A 3x3 filter uses the exact same 9 weights across the entire image, drastically reducing the number of parameters.
   2) **Local Connectivity**: They preserve the spatial relationship of pixels, whereas flattening destroys it.
2. **If you want the output spatial dimensions to exactly match the input spatial dimensions when using a 3x3 kernel with stride 1, what padding should you use?**
   *Answer*: `padding=1`.

## 19. Day Summary
- Convolutions slide a learned kernel (filter) across an image to detect patterns.
- Without padding, convolutions shrink the image.
- Max Pooling is used to aggressively downsample the image, reducing computation and adding translation invariance.